In [6]:
import pandas as pd

# Load dataset (simple)
df = pd.read_csv("books_clean.csv")

# Clean column names
df.columns = df.columns.str.strip()

# Get top 60 genres
top_60 = df["primary_genre"].value_counts().head(60).index

# Filter dataset
df_top60 = df[df["primary_genre"].isin(top_60)]

# Reset index
df_top60 = df_top60.reset_index(drop=True)

# Save dataset
df_top60.to_csv("books_60_genres_final.csv", index=False)

# Print summary
print("Total rows:", len(df_top60))
print("Unique genres:", df_top60["primary_genre"].nunique())



Total rows: 11697
Unique genres: 60


In [8]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

# --------------------------------------------------
# 1. Load dataset
# --------------------------------------------------
df = pd.read_csv("books_60_genres_final.csv")

# Keep only required columns
df = df[["plot_clean", "primary_genre"]]

# Remove missing values if any
df = df.dropna()

# Remove empty text rows if any
df = df[df["plot_clean"].astype(str).str.strip() != ""]

# --------------------------------------------------
# 2. Input and output
# --------------------------------------------------
X = df["plot_clean"]
y = df["primary_genre"]

# Encode target labels into numbers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# --------------------------------------------------
# 3. Train-test split
# --------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# --------------------------------------------------
# 4. Function to train and evaluate model
# --------------------------------------------------
def run_model(model_name, pipeline, param_grid):
    print("\n" + "=" * 60)
    print("Model:", model_name)
    print("=" * 60)

    grid = GridSearchCV(
        pipeline,
        param_grid,
        cv=3,
        scoring="accuracy",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)

    print("Best Parameters:", grid.best_params_)
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)

    return {
        "Model": model_name,
        "Best Parameters": grid.best_params_,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec
    }

# --------------------------------------------------
# 5. Decision Tree with Bag of Words
# --------------------------------------------------
dt_pipeline = Pipeline([
    ("bow", CountVectorizer()),
    ("clf", DecisionTreeClassifier(random_state=42))
])

dt_param_grid = {
    "bow__max_features": [3000, 5000],
    "bow__ngram_range": [(1, 1), (1, 2)],
    "clf__criterion": ["gini", "entropy"],
    "clf__max_depth": [20, 40, None],
    "clf__min_samples_split": [2, 5]
}

# --------------------------------------------------
# 6. KNN with Bag of Words
# --------------------------------------------------
knn_pipeline = Pipeline([
    ("bow", CountVectorizer()),
    ("clf", KNeighborsClassifier())
])

knn_param_grid = {
    "bow__max_features": [3000, 5000],
    "bow__ngram_range": [(1, 1), (1, 2)],
    "clf__n_neighbors": [3, 5, 7],
    "clf__weights": ["uniform", "distance"],
    "clf__metric": ["cosine", "minkowski"]
}

# --------------------------------------------------
# 7. Naive Bayes with Bag of Words
# --------------------------------------------------
nb_pipeline = Pipeline([
    ("bow", CountVectorizer()),
    ("clf", MultinomialNB())
])

nb_param_grid = {
    "bow__max_features": [3000, 5000],
    "bow__ngram_range": [(1, 1), (1, 2)],
    "clf__alpha": [0.1, 0.5, 1.0]
}

# --------------------------------------------------
# 8. Run all models
# --------------------------------------------------
results = []

results.append(run_model("Decision Tree + Bag of Words", dt_pipeline, dt_param_grid))
results.append(run_model("KNN + Bag of Words", knn_pipeline, knn_param_grid))
results.append(run_model("Naive Bayes + Bag of Words", nb_pipeline, nb_param_grid))

# --------------------------------------------------
# 9. Final results table
# --------------------------------------------------
results_df = pd.DataFrame(results)

print("\nFinal Comparison Table:")

# --------------------------------------------------
# Clean Results Table Printer
# Add this at the END of your existing code,
# replacing the final print and to_csv section
# --------------------------------------------------

# 1. Simplified results table (no Best Parameters column — too wide to print)
summary_df = pd.DataFrame([{
    "Model"    : r["Model"],
    "Accuracy" : round(r["Accuracy"],  4),
    "Precision": round(r["Precision"], 4),
    "Recall"   : round(r["Recall"],    4)
} for r in results])

# 2. Print clean aligned table
print("\n")
print("=" * 62)
print(f"{'FINAL RESULTS — BAG OF WORDS':^62}")
print("=" * 62)
print(f"{'Model':<32} {'Accuracy':>8} {'Precision':>10} {'Recall':>8}")
print("-" * 62)
for _, row in summary_df.iterrows():
    print(f"{row['Model']:<32} {row['Accuracy']:>8.4f} {row['Precision']:>10.4f} {row['Recall']:>8.4f}")
print("=" * 62)

# 3. Print best parameters separately, one model at a time
print("\nBEST PARAMETERS PER MODEL")
print("=" * 62)
for r in results:
    print(f"\nModel : {r['Model']}")
    print(f"{'Parameter':<35} {'Value'}")
    print("-" * 62)
    for k, v in r["Best Parameters"].items():
        print(f"  {k:<33} {v}")

# 4. Save clean CSV
summary_df.to_csv("bow_model_results.csv", index=False)
print("\n✅ Results saved to bow_model_results.csv")

# Save results table
results_df.to_csv("bow_model_results.csv", index=False)


Model: Decision Tree + Bag of Words
Best Parameters: {'bow__max_features': 5000, 'bow__ngram_range': (1, 1), 'clf__criterion': 'gini', 'clf__max_depth': 20, 'clf__min_samples_split': 2}
Accuracy : 0.2623931623931624
Precision: 0.22899909882885403
Recall   : 0.2623931623931624

Model: KNN + Bag of Words
Best Parameters: {'bow__max_features': 5000, 'bow__ngram_range': (1, 1), 'clf__metric': 'cosine', 'clf__n_neighbors': 7, 'clf__weights': 'distance'}
Accuracy : 0.2482905982905983
Precision: 0.24471235942028488
Recall   : 0.2482905982905983

Model: Naive Bayes + Bag of Words
Best Parameters: {'bow__max_features': 5000, 'bow__ngram_range': (1, 1), 'clf__alpha': 0.1}
Accuracy : 0.43205128205128207
Precision: 0.4247358537120119
Recall   : 0.43205128205128207

Final Comparison Table:


                 FINAL RESULTS — BAG OF WORDS                 
Model                            Accuracy  Precision   Recall
--------------------------------------------------------------
Decision Tree + Bag o